In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
chexpert = pd.read_csv('../data/raw/mimic-cxr-2.0.0-chexpert.csv.gz')
metadata = pd.read_csv('../data/raw/mimic-cxr-2.0.0-metadata.csv.gz')
split = pd.read_csv('../data/raw/mimic-cxr-2.0.0-split.csv.gz')

In [ ]:
# MIMIC-CXR-JPG CSV 데이터 구조 확인
# - CheXpert, Metadata, Split 데이터의 행·열 크기 확인
# - 각 데이터의 컬럼명 및 상위 5개 행 확인
# - 데이터 구성과 주요 변수 파악

for name, df in [
    ("CheXpert", chexpert),
    ("Metadata", metadata),
    ("Split", split)
]:
    print(f"\n===== {name} =====")
    print("데이터 크기:", df.shape)
    print("컬럼 목록:", df.columns.tolist())
    display(df.head())


===== CheXpert =====
데이터 크기: (227827, 16)
컬럼 목록: ['subject_id', 'study_id', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices']


,subject_id,study_id,Atelectasis,Cardiomegaly,Consolidation,Edema,Enlarged Cardiomediastinum,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices
0,10000032,50414267,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
1,10000032,53189527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
2,10000032,53911762,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
3,10000032,56699142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
4,10000764,57375967,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN



===== Metadata =====
데이터 크기: (377110, 12)
컬럼 목록: ['dicom_id', 'subject_id', 'study_id', 'PerformedProcedureStepDescription', 'ViewPosition', 'Rows', 'Columns', 'StudyDate', 'StudyTime', 'ProcedureCodeSequence_CodeMeaning', 'ViewCodeSequence_CodeMeaning', 'PatientOrientationCodeSequence_CodeMeaning']


,dicom_id,subject_id,study_id,PerformedProcedureStepDescription,ViewPosition,Rows,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning
0,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,10000032,50414267,CHEST (PA AND LAT),PA,3056,2544,21800506,213014.531,CHEST (PA AND LAT),postero-anterior,Erect
1,174413ec-4ec4c1f7-34ea26b7-c5f994f8-79ef1962,10000032,50414267,CHEST (PA AND LAT),LATERAL,3056,2544,21800506,213014.531,CHEST (PA AND LAT),lateral,Erect
2,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,10000032,53189527,CHEST (PA AND LAT),PA,3056,2544,21800626,165500.312,CHEST (PA AND LAT),postero-anterior,Erect
3,e084de3b-be89b11e-20fe3f9f-9c8d8dfe-4cfd202c,10000032,53189527,CHEST (PA AND LAT),LATERAL,3056,2544,21800626,165500.312,CHEST (PA AND LAT),lateral,Erect
4,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,10000032,53911762,CHEST (PORTABLE AP),AP,2705,2539,21800723,80556.875,CHEST (PORTABLE AP),antero-posterior,NaN



===== Split =====
데이터 크기: (377110, 4)
컬럼 목록: ['dicom_id', 'study_id', 'subject_id', 'split']


,dicom_id,study_id,subject_id,split
0,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,50414267,10000032,train
1,174413ec-4ec4c1f7-34ea26b7-c5f994f8-79ef1962,50414267,10000032,train
2,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,53189527,10000032,train
3,e084de3b-be89b11e-20fe3f9f-9c8d8dfe-4cfd202c,53189527,10000032,train
4,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,53911762,10000032,train


In [ ]:
# MIMIC-CXR-JPG 데이터별 고유 식별자 수 확인
# - subject_id: 고유 환자 수
# - study_id: 고유 검사 수
# - dicom_id: 고유 X-ray 이미지 수
# - CheXpert, Metadata, Split 간 데이터 규모 비교
for name, df in [
    ("CheXpert", chexpert),
    ("Metadata", metadata),
    ("Split", split)
]:
    print(f"\n===== {name} =====")

    for col in ["subject_id", "study_id", "dicom_id"]:
        if col in df.columns:
            print(f"{col}: {df[col].nunique():,}")


===== CheXpert =====
subject_id: 65,379
study_id: 227,827

===== Metadata =====
subject_id: 65,379
study_id: 227,835
dicom_id: 377,110

===== Split =====
subject_id: 65,379
study_id: 227,835
dicom_id: 377,110


In [13]:
# 심비대(Cardiomegaly)·폐부종(Edema)·흉수(Pleural Effusion)
chexpert['Cardiomegaly'].info()
chexpert['Edema'].info()
chexpert['Pleural Effusion'].info()


<class 'pandas.Series'>
RangeIndex: 227827 entries, 0 to 227826
Series name: Cardiomegaly
Non-Null Count  Dtype  
--------------  -----  
66799 non-null  float64
dtypes: float64(1)
memory usage: 1.7 MB
<class 'pandas.Series'>
RangeIndex: 227827 entries, 0 to 227826
Series name: Edema
Non-Null Count  Dtype  
--------------  -----  
65833 non-null  float64
dtypes: float64(1)
memory usage: 1.7 MB
<class 'pandas.Series'>
RangeIndex: 227827 entries, 0 to 227826
Series name: Pleural Effusion
Non-Null Count  Dtype  
--------------  -----  
87272 non-null  float64
dtypes: float64(1)
memory usage: 1.7 MB


In [ ]:
# 주요 3개 소견별 레이블 분포 확인
# - 분석 대상: 심비대(Cardiomegaly), 폐부종(Edema), 흉수(Pleural Effusion)
# - 양성(1), 음성(0), 불확실(-1), 결측(NaN) 개수 확인
# - 결측값을 포함하여 각 소견의 레이블 분포 파악
target_labels = [
    "Cardiomegaly",
    "Edema",
    "Pleural Effusion"
]

# 각 레이블의 양성, 음성, 불확실, 결측 분포
for label in target_labels:
    print(f"\n===== {label} =====")
    print(
        chexpert[label].value_counts(
            dropna=False
        )
    )


===== Cardiomegaly =====
Cardiomegaly
 NaN    161028
 1.0     44845
 0.0     15911
-1.0      6043
Name: count, dtype: int64

===== Edema =====
Edema
 NaN    161994
 1.0     27018
 0.0     25641
-1.0     13174
Name: count, dtype: int64

===== Pleural Effusion =====
Pleural Effusion
 NaN    140555
 1.0     54300
 0.0     27158
-1.0      5814
Name: count, dtype: int64


In [ ]:
# 주요 3개 소견별 레이블 분포표 생성
# - 분석 대상: 심비대, 폐부종, 흉수
# - 양성(1), 음성(0), 불확실(-1), 결측(NaN) 개수 집계
# - 전체 검사 수 대비 레이블별 비율(%) 계산
# - 소견별 개수와 비율을 하나의 DataFrame으로 정리

target_labels = [
    "Cardiomegaly",
    "Edema",
    "Pleural Effusion"
]

label_names = {
    1.0: "Positive",
    0.0: "Negative",
    -1.0: "Uncertain",
    "NaN": "Missing"
}
results = []
for label in target_labels:
    counts = chexpert[label].value_counts(dropna=False)
    for value, name in label_names.items():
        if value == "NaN":
            count = chexpert[label].isna().sum()
        else:
            count = counts.get(value, 0)

        results.append({
            "Finding": label,
            "Label": name,
            "Count": count,
            "Percent": round(count / len(chexpert) * 100, 2)
        })
label_distribution= pd.DataFrame(results)
label_distribution

,Finding,Label,Count,Percent
0,Cardiomegaly,Positive,44845,19.68
1,Cardiomegaly,Negative,15911,6.98
2,Cardiomegaly,Uncertain,6043,2.65
3,Cardiomegaly,Missing,161028,70.68
4,Edema,Positive,27018,11.86
5,Edema,Negative,25641,11.25
6,Edema,Uncertain,13174,5.78
7,Edema,Missing,161994,71.10
8,Pleural Effusion,Positive,54300,23.83
9,Pleural Effusion,Negative,27158,11.92


In [24]:
label_distribution[
    label_distribution["Label"] == "Positive"
]
triple_positive = chexpert[
    (chexpert["Cardiomegaly"] == 1) &
    (chexpert["Edema"] == 1) &
    (chexpert["Pleural Effusion"] == 1)
].copy()

print("세 가지 소견 모두 양성인 검사 수:", len(triple_positive))

display(triple_positive[
    ["subject_id", "study_id",
     "Cardiomegaly", "Edema", "Pleural Effusion"]
].head(10))

세 가지 소견 모두 양성인 검사 수: 6323


,subject_id,study_id,Cardiomegaly,Edema,Pleural Effusion
19,10000980,58206436,1.0,1.0,1.0
38,10001884,50712381,1.0,1.0,1.0
55,10001884,56722923,1.0,1.0,1.0
140,10003400,50183258,1.0,1.0,1.0
143,10003400,51345654,1.0,1.0,1.0
145,10003400,52181745,1.0,1.0,1.0
170,10003502,52139270,1.0,1.0,1.0
222,10007795,54054513,1.0,1.0,1.0
261,10010471,59553780,1.0,1.0,1.0
268,10010867,52872599,1.0,1.0,1.0


In [ ]:
# 주요 3개 소견의 동시 양성 검사 추출
# - 심비대, 폐부종, 흉수 각각의 양성(1) 분포 확인
# - 세 소견이 모두 양성(1)인 검사 추출
# - 해당 검사 수 및 상위 10개 검사 정보 확인

total = len(chexpert)
triple_count = len(triple_positive)

print(f"전체 검사 수: {total:,}")
print(f"세 소견 모두 양성: {triple_count:,}")
print(f"전체 대비 비율: {triple_count / total * 100:.2f}%")

print(
    "고유 환자 수:",
    triple_positive["subject_id"].nunique()
)

전체 검사 수: 227,827
세 소견 모두 양성: 6,323
전체 대비 비율: 2.78%
고유 환자 수: 3543


In [27]:
triple_positive.to_csv(
    "../results/tables/triple_positive.csv",
    index=False
)

In [ ]:
# 8가지 조합별 분포 
all_negative=chexpert[
    (chexpert['Cardiomegaly'] == 0) &
    (chexpert['Edema'] == 0) &
    (chexpert['Pleural Effusion'] == 0)
]
Cardiomegaly_only = chexpert[
    (chexpert['Cardiomegaly'] == 1) &
    (chexpert['Edema'] == 0) &
    (chexpert['Pleural Effusion'] == 0)
]
Edema_only = chexpert[
    (chexpert['Cardiomegaly'] == 0) &
    (chexpert['Edema'] == 1) &
    (chexpert['Pleural Effusion'] == 0)
]
Pleural_Effusion_only = chexpert[
    (chexpert['Cardiomegaly'] == 0) &
    (chexpert['Edema'] == 0) &
    (chexpert['Pleural Effusion'] == 1)
]
Cardiomegaly_Edema = chexpert[
    (chexpert['Cardiomegaly'] == 1) &
    (chexpert['Edema'] == 1) &
    (chexpert['Pleural Effusion'] ==0)
]
Cardiomegaly_Pleural_Effusion=chexpert[
    (chexpert['Cardiomegaly'] == 1) &
    (chexpert['Edema'] == 0) &
    (chexpert['Pleural Effusion'] ==1)
]
Edema_Pleural_Effusion=chexpert[
    (chexpert['Cardiomegaly'] == 0) &
        (chexpert['Edema'] == 1) &
        (chexpert['Pleural Effusion'] ==1)

]
all_positive = chexpert[
    (chexpert['Cardiomegaly'] == 1) &
    (chexpert['Edema'] == 1) &
    (chexpert['Pleural Effusion'] == 1)
]

In [ ]:
# 8가지 조합별 분포표

combined_distribution = pd.DataFrame({
    'Combination': [
        'All Negative',
        'Cardiomegaly Only',
        'Edema Only',
        'Pleural Effusion Only',
        'Cardiomegaly + Edema',
        'Cardiomegaly + Pleural Effusion',
        'Edema + Pleural Effusion',
        'All Positive'
    ],
    'Count': [
        len(all_negative),
        len(Cardiomegaly_only),
        len(Edema_only),
        len(Pleural_Effusion_only),
        len(Cardiomegaly_Edema),
        len(Cardiomegaly_Pleural_Effusion),
        len(Edema_Pleural_Effusion),
        len(all_positive)
    ]
})

combined_distribution['Percent'] = (
    combined_distribution['Count'] / len(chexpert) * 100
).round(2)

display(combined_distribution)

,Combination,Count,Percent
0,All Negative,3319,1.46
1,Cardiomegaly Only,4506,1.98
2,Edema Only,276,0.12
3,Pleural Effusion Only,554,0.24
4,Cardiomegaly + Edema,1354,0.59
5,Cardiomegaly + Pleural Effusion,2557,1.12
6,Edema + Pleural Effusion,862,0.38
7,All Positive,6323,2.78
